# Notebook C - fourteen-slide preliminary discussion deck

Run the updated B first. Slide 7 contains annual participation and concentration charts; new slide 8 contains data-type evolution. A separate paper_replication_report.html is copied alongside the deck. No source acquisition or new analytical estimates occur in C.


Requires the fixed B output. Slide 4 distinguishes recorded labels from vocabulary validation. Slide 7 shows contributor/accessor columns and PI-ranked volumes. Still 14 slides; no acquisition occurs here.


In [ ]:
# @title Build fourteen preliminary-discussion slides (HTML + PowerPoint)
DRIVE_ROOT = '/content/drive/MyDrive/dbgap_analysis'
EXPORT_POWERPOINT = True
INSTALL_MISSING = True
MOUNT_DRIVE = True
DOWNLOAD_IN_COLAB = True
# Reads only the matching Notebook B export. No acquisition or analytical recomputation.

import sys, os, json, hashlib, html, math, re, io, base64, textwrap, importlib.util, subprocess
from pathlib import Path
from datetime import datetime, timezone
if MOUNT_DRIVE and DRIVE_ROOT.startswith('/content/drive/'):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError: pass
needed=[]
if EXPORT_POWERPOINT:
    for module,package in [('pptx','python-pptx'),('cairosvg','cairosvg')]:
        if importlib.util.find_spec(module) is None:needed.append(package)
if needed and INSTALL_MISSING:subprocess.run([sys.executable,'-m','pip','install','-q',*needed],check=True)
BASE=Path(DRIVE_ROOT)/'preliminary14_analysis'
if not (BASE/'summary.json').exists():raise FileNotFoundError('Run the new Notebook B first: '+str(BASE/'summary.json'))
def digest(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for chunk in iter(lambda:f.read(1048576),b''):h.update(chunk)
    return h.hexdigest()
MANIFEST=json.loads((BASE/'manifest.json').read_text())
for name,sha in MANIFEST['outputs'].items():
    p=(BASE/name).resolve()
    if not p.is_relative_to(BASE.resolve()):raise ValueError('Unsafe manifest path: '+name)
    if not p.exists() or digest(p)!=sha:raise ValueError('B output missing or changed: '+name+'; rerun Notebook B.')
R=json.loads((BASE/'summary.json').read_text())
if R.get('version')!='preliminary-fourteen-contributors-accessors-v2':raise ValueError('Rerun the fixed Notebook B before C: contributor ranks and semantic readiness fields are required.')
if R.get('input_signature')!=MANIFEST.get('signature'):raise ValueError('B summary/manifest signatures differ.')
# Current source files are checked where accessible. Do not present stale B products after A changes.
for source in MANIFEST.get('inputs',[]):
    p=Path(source['path'])
    if not p.exists():raise FileNotFoundError('B input unavailable: '+str(p))
    if digest(p)!=source['sha256']:raise ValueError('Source changed since B ran: '+str(p))
OUT=Path(DRIVE_ROOT)/'preliminary14_decks'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT.mkdir(parents=True,exist_ok=False)
W,H=1440,810
C={'ink':'#183047','blue':'#235A85','teal':'#137D83','amber':'#BA7D24','muted':'#607487','light':'#EEF4F7','line':'#D8E3EA','white':'#FFFFFF'}
SLIDES=[]

def esc(x):return html.escape(str(x),quote=True)
def integer(x):return 'Not measured' if x is None else f'{int(x):,}'
def percent(x):return '--' if x is None else f'{x:.1f}%'
def dec(x,d=3):return '--' if x is None else f'{x:.{d}f}'
def add(title,subtitle='',foot='',notes='',refs=None):
    s={'title':title,'subtitle':subtitle,'foot':foot,'notes':notes,'refs':refs or [],'elements':[]}
    SLIDES.append(s)
    rect(s,0,0,W,H,C['white']);rect(s,48,36,64,5,C['teal'])
    txt(s,title,48,52,1344,47,32,bold=True)
    if subtitle:txt(s,subtitle,48,103,1344,33,18,C['muted'])
    if R.get('label'):txt(s,R['label'],1010,21,382,23,13,C['amber'],True,align='right')
    rect(s,48,749,1344,1,C['line'])
    txt(s,foot,48,760,1260,34,12,C['muted'])
    txt(s,f'{len(SLIDES):02d} / 14',1315,760,78,22,14,C['muted'],align='right')
    return s
def rect(s,x,y,w,h,fill,border=None):s['elements'].append({'kind':'rect','x':x,'y':y,'w':w,'h':h,'fill':fill,'border':border})
def txt(s,text,x,y,w,h,size=20,color=None,bold=False,align='left',link=None):
    s['elements'].append({'kind':'text','text':str(text),'x':x,'y':y,'w':w,'h':h,'size':size,'color':color or C['ink'],'bold':bold,'align':align,'link':link})
def card(s,value,label,x,y,w,h=101,sub=''):
    rect(s,x,y,w,h,C['light']);rect(s,x,y,4,h,C['teal'])
    txt(s,value,x+17,y+11,w-30,40,34,bold=True)
    txt(s,label,x+17,y+53,w-30,28,18,bold=True)
    if sub:txt(s,sub,x+17,y+81,w-30,27,13,C['muted'])
def figure(s,key,x,y,w,h):
    entry=R.get('figures',{}).get(key,{})
    path=BASE/entry.get('svg','__missing__')
    if path.exists():s['elements'].append({'kind':'image','x':x,'y':y,'w':w,'h':h,'path':str(path)})
    else:
        rect(s,x,y,w,h,C['light']);txt(s,'Not measured from the available inputs',x+25,y+h/2-35,w-50,90,22,C['muted'],align='center')
def svg_image(s,svg,x,y,w,h,name):
    path=OUT/(name+'.svg');path.write_text(svg)
    s['elements'].append({'kind':'image','x':x,'y':y,'w':w,'h':h,'path':str(path)})
def table(s,headers,rows,x,y,widths,rowh=41,font=18):
    for j,title in enumerate(headers):
        xx=x+sum(widths[:j]);rect(s,xx,y,widths[j],rowh,C['blue']);txt(s,title,xx+10,y+7,widths[j]-20,rowh-10,font,C['white'],True)
    for i,row in enumerate(rows):
        yy=y+(i+1)*rowh
        for j,val in enumerate(row):
            xx=x+sum(widths[:j]);rect(s,xx,yy,widths[j],rowh,C['light'] if i%2==0 else C['white'])
            txt(s,val,xx+10,yy+7,widths[j]-20,rowh-8,font,align='left')
def note(s,text,y=696,size=17):
    rect(s,48,y,1344,43,'#EEF7F6');txt(s,text,61,y+9,1318,34,size,C['ink'])
def ref_links(s,keys,y):
    refs={r['key']:r for r in R.get('references',[])};x=48
    for key in keys:
        ref=refs.get(key)
        if not ref:continue
        label=ref['label'];width=min(445,10*len(label)+20)
        txt(s,label,x,y,width,24,14,C['blue'],link=ref['url']);x+=width+15

o=R['overview'];mode=R.get('mode','saved_table_analysis')
source_label='REPORTED SNAPSHOT - NOT RECOMPUTED' if mode=='reported_snapshot' else 'Saved-data analysis'
if R.get('label'):source_label=R['label']+' | '+source_label
N=o['accessions']
# 1
s=add('dbGaP overview numbers','Preliminary discussion | recorded metadata, not a count of verified reuse',
    f'{source_label}. Root accession universe: {integer(N)}. Unknown source states are retained.',
    notes='Publications are distinct successfully retrieved linked PMIDs; unresolved candidate PMIDs are not counted as confirmed publications. Clinical trials are distinct NCT identifiers, not NCT-positive roots. DAR total is the captured accession-requester row count, not distinct consent-group applications. NIH acknowledgement means an explicit NIH/IC mention in public study funding/acknowledgement sections; it does not use RePORTER as a proxy. SSTR means a successfully retrieved current summary.')
for j,(val,label,sub) in enumerate([(o['accessions'],'Root accessions','Not independent cohorts'),(o['publications'],'PubMed publications','Distinct retrieved PMIDs'),(o['dars'],'Observed DAR records','Accession-requester rows'),(o['clinical_trials'],'Linked clinical trials','Distinct NCT identifiers')]):
    card(s,integer(val),label,48+341*j,153,321,117,sub)
rows=[[r['label'],integer(r['positive']),percent(r['percentage_all_roots']),f"{r['known']:,} / {r['unknown']:,}"] for r in o['coverage']]
table(s,['Accessions with ...','Number','% of all accessions','Known / unknown'],rows,48,302,[595,180,245,324],46,19)
note(s,'Different denominators answer different questions: roots, papers, access records and trial identifiers.',694)
# 2. Growth: six bar charts, all on the same 2006-2026 x axis.
s=add('Growth','Annual counts (top row) and cumulative counts (bottom row) | identical 2006-2026 calendar axes',
    f'{source_label}. Dated subsets only. 2026 may be incomplete; no annualization.',
    notes=R['growth']['accession_note']+' Cumulative bars include valid observations before 2006; the pre-window baseline is displayed. Publication years describe the currently linked bibliography, not historical repository inventories. Rows after a known observation cutoff remain blank, not zero. Unavailable dates are not replaced with first DAR or publication dates.')
for j,(key,title) in enumerate([('accessions','Accessions'),('publications','Publications'),('dars','DAR records')]):
    x=48+j*453;g=R['growth'][key]
    txt(s,title,x,150,426,29,23,bold=True)
    figure(s,'growth_'+key+'_annual',x,183,426,219)
    figure(s,'growth_'+key+'_cumulative',x,415,426,219)
    txt(s,f"{integer(g['n'])} dated | {integer(g.get('before_window',0))} before 2006",x,642,426,26,15,C['muted'])
note(s,'Cumulative includes the pre-2006 baseline. Accession date = '+('retained FTP-artifact timestamp, not release.' if R['growth']['accession_basis']=='ftp_artifact' else 'first local observation, not historical release.'),687,15)

# 3. Focused profiles, named consents and actual v/p suffix coordinates.
p=R['profile'];v=p['versions'];cons=p['consents']
s=add('Accession size, consent and versions','Subjects, samples and variables | named consents | data version x participant set',
    f'{source_label}. One root per numeric profile/matrix; consent categories may overlap. Missing values are excluded, not zero.',
    notes='\n'.join(x['label']+': '+x['definition'] for x in p['definitions'])+'\n'+p['notes']+
    '\nVersion diagnostic: '+json.dumps(v,indent=2)+
    '\nA singleton retained v10.p1 record has coordinates (10,1), not (1,1). The matrix is log-colored and explicitly annotated so v1.p1 cannot hide the tail. The matrix uses highest observed co-occurring pairs; it never combines separate maxima. It is not a complete historical release census.'+
    '\nConsent code-to-name mappings are local to each accession/version. Example: code 1 is GRU in some studies and HMB in others. Modifiers are retained. Full lookup and frequency tables are exported by B.\nAll consent labels: '+json.dumps(cons['rows'],indent=2))
for j,key in enumerate(['subjects','samples','variables']):
    figure(s,'profile_'+key,48+j*453,147,430,215)
figure(s,'consent_labels',48,376,649,267)
figure(s,'version_matrix',729,376,663,267)
h=p['hierarchy']
items=[('Parent accessions',h['parents'],f"{h['parent_known']:,} with child-count evidence"),
       ('Child accessions',h['children'],f"{h['child_known']:,} with parent evidence"),
       ('Parent-child pairs',h['pairs'],f"{h['pairs_both_in_universe']:,} within root universe"),
       ('Collections',h['collections'],f"{h['collection_known']:,} known; {h['unknown_collections']:,} unknown")]
for j,(label,val,sub) in enumerate(items):
    x=48+j*341;rect(s,x,660,321,76,C['light'])
    txt(s,integer(val) if val is not None else '--',x+12,668,98,30,25,bold=True)
    txt(s,label,x+112,672,201,25,17,bold=True)
    txt(s,sub,x+12,708,298,20,12,C['muted'])

# Slide 4: separate source coverage, vocabulary validation, and label harmonization.
sem=R.get('semantics',{});mesh_ok=bool(sem.get('mesh_vocabulary_available'));obi_ok=bool(sem.get('obi_vocabulary_available'))
s=add('Semantic profiles and reported data types','Source coverage, ontology readiness and the recorded assay mix',
    f"{source_label}. One root per category; labels overlap. Native annotation is distinct from ontology validation.",
    notes=sem.get('note','')+'\n'+json.dumps(sem,indent=2)+
    '\nNative source labels are retained when a vocabulary is unavailable. The fallback chart reports those labels only; no hierarchy, synonym equivalence or new subtype is inferred. No terms are copied from trials or linked articles.'+
    '\nAn ontology download or parsing failure is not an observed lack of MeSH coverage. Local assay families are a transparent crosswalk, not OBI validation when the OBI snapshot is missing.'+
    '\nFor fully resolved profiles, specific terms suppress redundant parents only in the display. All source annotations and tree paths remain in B exports. Project and trial references do not establish a causal pathway.')
card(s,integer(sem.get('roots_with_recorded_mesh')),'Native MeSH recorded',48,145,432,109,'Source annotations; no terms imputed')
card(s,integer(sem.get('roots_with_native_mesh')) if mesh_ok else 'Not run','MeSH vocabulary validation',502,145,432,109,
     (integer(sem.get('roots_with_redundant_broad_terms_refined'))+' roots with supported refinement') if mesh_ok else 'Snapshot unavailable; native labels retained')
card(s,integer(sem.get('roots_with_mapped_assay')),'Reported assay families',956,145,432,109,'Local label crosswalk; not file-level harmonization')
figure(s,'semantic_themes',48,270,654,286);figure(s,'semantic_assays',735,270,657,286)
rect(s,48,578,654,111,C['light']);rect(s,735,578,657,111,C['light'])
txt(s,'Vocabulary status',63,590,620,25,20,bold=True)
statusrows=sem.get('ontology_status',{})
mesh_status=next((str(v) for k,v in statusrows.items() if k.startswith('mesh_')),'not supplied')
txt(s,'MeSH: '+('loaded' if mesh_ok else 'unavailable ('+mesh_status+')')+'\nOBI: '+('loaded' if obi_ok else 'unavailable; assay labels use the local crosswalk'),63,625,620,53,17,C['muted'])
txt(s,'Dataset-trial semantic context',750,590,620,25,20,bold=True)
if mesh_ok:
    ctx=(f"{integer(sem.get('native_exact_pairs'))} exact; {integer(sem.get('hierarchy_additional_pairs'))} additional hierarchy matches.\n"
         f"{integer(sem.get('pairs_with_resolved_both_sides'))}/{integer(sem.get('pair_total'))} pairs have both sides resolved.")
else:ctx='Comparison not evaluated without the MeSH snapshot.\nZero resolved matches is not evidence of no relationship.'
txt(s,ctx,750,625,620,53,17,C['muted'])
ref_links(s,['mesh_semantics','obi_semantics','rcdc_semantics'],706)

# 5
m=R['matrix'];s=add('DARs and publications: overlap and intensity',f"2 x 2 among {m['known']:,} accessions with both signals known; {m['unknown']:,} unknown excluded",
    f'{source_label}. Matrix percentages use the doubly known subset. Histograms use complete recorded counts.',
    notes='A negative state means no recorded signal in a supported extraction, not no use. Histogram bins include zero and the entire right tail. Publications count unique successful PMIDs per root; missing counts are excluded.')
for j,row in enumerate(m['rows']):
    x=48+(j%2)*309;y=166+(j//2)*220
    card(s,integer(row['count']),row['label'],x,y,289,185,percent(row['percent_known'])+' of known roots')
figure(s,'paper_hist',696,150,696,247);figure(s,'dar_hist',696,425,696,247)
note(s,'Publication without a DAR may reflect original output, earlier access, incomplete capture, or a misleading link.')
# 5. Replace the unestimated-model panel with an interpretable descriptive comparison.
g=R['group_comparison'];pop=g['population']
s=add('DAR + publications versus neither','Descriptive comparison and top five MeSH terms within each group',
    f'{source_label}. MeSH % uses all roots in each group; terms overlap. Mixed/unknown signals excluded only from this comparison.',
    notes=g['note']+'\n'+g['mesh_basis']+'\nFull descriptive statistics: '+json.dumps(g['summary'],indent=2)+
    '\nMeSH term tables: '+json.dumps(g['mesh_groups'],indent=2)+
    '\nThe prior unestimated regression panel has been removed. Model fitting is optional/off in B and its diagnostic output is not a result on this slide. There is no adjusted or causal interpretation. Missingness, study age, program composition and linkage quality can differ between groups.')
for j,(label,val,sub) in enumerate([
        ('DAR + publication',pop['both'],'Both captured signals'),('Neither',pop['neither'],'Neither captured signal'),
        ('Outside this comparison',pop['mixed_excluded']+pop['unknown_excluded'],f"{pop['mixed_excluded']:,} mixed; {pop['unknown_excluded']:,} unknown")]):
    card(s,integer(val),label,48+j*454,145,432,111,sub)
def descript_cell(feature,target):
    q=next((r for r in g['summary'] if r['feature']==feature and r['group']==target),None)
    if not q or q['known']==0:return 'Not measured'
    fmt=lambda x: f'{x:,.0f}' if float(x).is_integer() else f'{x:,.1f}'
    return f"{fmt(q['median'])} [{fmt(q['p25'])}-{fmt(q['p75'])}]; n={q['known']:,}"
rows=[[label,descript_cell(key,1),descript_cell(key,0)] for key,label in [('subjects','Subjects'),('samples','Samples'),('variables','Variables'),('consent_codes','Consent-code count')]]
table(s,['Per-accession median [IQR]; measured n','DAR + publication','Neither'],rows,48,278,[490,427,427],31,16)
for j,group in enumerate(g['mesh_groups']):
    x=48+j*686
    txt(s,'Top 5 MeSH: '+group['label'],x,452,657,31,22,bold=True)
    txt(s,f"Named MeSH: {group['with_named_mesh']:,}/{group['n']:,} roots; {group['without_named_mesh']:,} without labels",x,487,657,22,14,C['muted'])
    headers=['MeSH descriptor','Roots','% group'];widths=[425,83,150]
    rows=[]
    for term in group['top5']:
        label=term['label']
        if len(label)>48:label=label[:45]+'...'
        rows.append([label,integer(term['accessions']),percent(term['percentage_group'])])
    if rows:table(s,headers,rows,x,517,widths,35,15)
    else:
        rect(s,x,517,658,170,C['light'])
        txt(s,'No named MeSH descriptors available for this group.\nNo terms inferred from titles.',x+18,563,620,83,19,C['muted'])

# Slide 7: two distinct roles, with like-for-like annual panels and independent ranks.
part=R['participation'];a=part['annual'];con=part['contributor_volume'];proxy=bool(con['attribution_proxy'])
s=add('Data contributors versus data accessors','Annual participation above; contributed accessions and DAR requests ranked across all PI / group labels below',
    source_label+'. Contributors and accessors may overlap. Counts use different units; no contributor/accessor ratio is implied.',
    notes=part['notes']+'\n'+json.dumps(part,indent=2)+
    '\nLeft ranks count distinct root-PI/group relationships, not files or upload events. Public study attribution is a proxy unless verified submission records are supplied. Repeated versions and evidence lines do not inflate contributor counts.'+
    '\nRight ranks count distinct root-WLID relationships grouped by name plus organization, with WLID fallback. These are neither adjudicated people nor completed data uses. All entities are shown in descending order, not just a top-ten subset.'+
    '\nAnnual x axes remain 2006-2026; current year may be incomplete. Current attribution by discovery cohort is not historical submission activity.')
rect(s,48,146,654,44,C['teal']);rect(s,735,146,657,44,C['blue'])
txt(s,'DATA CONTRIBUTORS',63,154,620,30,23,C['white'],True)
txt(s,'DATA ACCESSORS',750,154,620,30,23,C['white'],True)
for key,x,y in [('submitter_people',48,200),('requester_people',735,200),('submitter_institutions',48,367),('requester_institutions',735,367)]:
    figure(s,'annual_'+key,x,y,657 if x==735 else 654,155)
figure(s,'rank_contributors_pi',48,534,654,155)
figure(s,'rank_accessors_pi',735,534,657,155)
left='Contributor proxy: public PI attribution by discovery cohort.' if proxy else 'Contributor records: validated data-submitter attribution.'
right='Accessors: initial approvals; PI-name/organization proxies.'
txt(s,left,48,700,654,38,15,C['teal'],True)
txt(s,right,735,700,657,38,15,C['blue'],True)

# New slide 8. Temporal composition of reported data types.
dt=R['data_type_evolution']
s=add('Data types: composition and evolution','Current families, breadth per accession, annual counts and within-cohort shares',
    source_label+'. Categories overlap; sample-assay counts are not unique biological samples.',
    notes=dt['note']+'\nTime basis: '+dt['basis']+'\n'+json.dumps(dt,indent=2))
figure(s,'type_current',48,157,653,246);figure(s,'type_breadth',735,157,657,246)
figure(s,'type_annual',48,416,653,259);figure(s,'type_share',735,416,657,259)
txt(s,f"{integer(dt['annotated_roots'])} annotated roots; {integer(dt['dated_roots'])} dated. {dt['basis']}.",48,694,1344,39,15,C['amber'],True)

# 7. Keep both requested histograms and add the top-accession detail tables.
rcr=R['rcr'];leaders=R['top_accessions']
s=add('Paper RCRs and the highest-activity accessions','Two distributions, followed by separate top-five lists for recorded DARs and linked papers',
    f"{source_label}. {integer(rcr['n'])} scored PMIDs. Rankings are by observed counts; >= denotes an incomplete PubMed count.",
    notes='The left plot repeats the paper-count distribution from slide 4. The RCR plot counts each scored PMID once. Zero scores are retained; missing scores are excluded. Unequal-width bins are count categories, not densities.\n'+
    leaders['note']+'\n'+leaders['dar_definition']+'\n'+leaders['pubmed_definition']+
    '\nTop lists with full names, links and counts: '+json.dumps(leaders['groups'],indent=2)+
    '\nA paper can link to more than one accession. Root rankings are not a ranking of independent cohorts or causal sharing impact. Counts at the fifth-place tie are ordered by accession, not manually selected.')
figure(s,'paper_hist',48,151,650,224);figure(s,'rcr_hist',742,151,650,224)
rect(s,48,383,1344,40,C['light'])
txt(s,f"Median RCR: {dec(rcr['median'],2)}    |    Zero-RCR papers: {integer(rcr['zero'])}    |    Top 10% share of summed RCR: "+
    (percent(100*rcr['top10_weighted_share']) if rcr['top10_weighted_share'] is not None else '--'),61,393,1310,24,17,bold=True)
for j,group in enumerate(leaders['groups']):
    x=48+j*686;rows=group['rows'];widths=[464,86,108]
    txt(s,group['title'],x,442,658,27,21,bold=True)
    y=478
    for col,(head,ww) in enumerate(zip(['Accession / study','DARs','PubMed'],widths)):
        xx=x+sum(widths[:col]);rect(s,xx,y,ww,32,C['blue']);txt(s,head,xx+9,y+7,ww-17,24,15,C['white'],True)
    if not rows:
        rect(s,x,y+33,658,165,C['light']);txt(s,'No positive accession counts available for this ranking.',x+18,y+68,620,72,20,C['muted'])
    for k,row in enumerate(rows):
        yy=y+33+k*43
        rect(s,x,yy,658,42,C['light'] if k%2==0 else C['white'])
        title=row['study_title'];title=title if len(title)<=57 else title[:54]+'...'
        txt(s,row['root_accession']+' | '+title,x+8,yy+4,448,35,14,link=row['study_url'])
        txt(s,integer(row['dar_records']) if row['dar_records'] is not None else '--',x+473,yy+10,69,26,16,bold=True,align='right')
        pub=integer(row['pubmed_papers']) if row['pubmed_papers'] is not None else '--'
        if row['pubmed_count_lower_bound'] and row['pubmed_papers'] is not None:pub='>= '+pub
        txt(s,pub,x+553,yy+10,97,26,16,bold=True,align='right')

# 6: exactly conserved root-state Sankey, generated only from actual joint state counts.
def sankey_svg(data):
    width,height=1320,490;scale=292/max(1,data['n']);nodes={}
    xs=[104,380,656,932,1208]
    parts=[f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">']
    colors={'All':'#235A85','Yes':'#137D83','No':'#8C9BA8','Unknown':'#BA7D24'}
    for j in range(5):
        stage=[n for n in data['nodes'] if n['stage']==j];cursor=141
        heading=stage[0]['stage_label'] if stage else ''
        parts.append(f'<text x="{xs[j]}" y="26" text-anchor="middle" font-family="Arial" font-size="23" font-weight="bold" fill="#183047">{esc(heading)}</text>')
        for q,n in enumerate(stage):
            parts.append(f'<text x="{xs[j]}" y="{55+q*22}" text-anchor="middle" font-family="Arial" font-size="18" fill="{colors[n["state"]]}">{esc(n["state"])}: {n["n"]:,}</text>')
            hh=n['n']*scale;nodes[n['id']]={'x':xs[j]-8,'y':cursor,'h':hh,'state':n['state']};cursor+=hh+(19 if n['n'] else 0)
    so=Counter();to=Counter()
    for link in sorted(data['links'],key=lambda l:(l['source'],l['target'])):
        a=nodes[link['source']];b=nodes[link['target']];hh=link['n']*scale
        x1=a['x']+16;x2=b['x'];y1=a['y']+so[link['source']];y2=b['y']+to[link['target']]
        mid=(x1+x2)/2;so[link['source']]+=hh;to[link['target']]+=hh
        d=f'M{x1},{y1} C{mid},{y1} {mid},{y2} {x2},{y2} L{x2},{y2+hh} C{mid},{y2+hh} {mid},{y1+hh} {x1},{y1+hh} Z'
        parts.append(f'<path d="{d}" fill="{colors[a["state"]]}" fill-opacity=".29"><title>{esc(link["source"])} to {esc(link["target"])}: {link["n"]:,} roots</title></path>')
    for n in nodes.values():
        if n['h']>0:parts.append(f'<rect x="{n["x"]}" y="{n["y"]}" width="16" height="{max(.6,n["h"])}" fill="{colors[n["state"]]}"/>')
    parts.append('</svg>');return ''.join(parts)
from collections import Counter
s=add('How the recorded signals intersect','Sankey / alluvial view: accessions -> SSTR availability -> DAR -> PubMed -> NCT linkage',
    f'{source_label}. Band width = root accessions at every stage. Unknown states retained.',
    notes='This is a cross-sectional root-state display, not a causal or chronological pipeline. Every root appears once in each column. Adjacent links do not establish a requester-paper-trial causal chain. Full joint paths are exported by B. SSTR, DAR, publications and trials are existence states, not mixed event totals.')
svg_image(s,sankey_svg(R['sankey']),48,149,1344,490,'signal_sankey')
note(s,'One accession remains one accession throughout. Connections show co-occurrence, not scientific conversion.',685)
# 7
v=R['venn']
def venn_svg(v):
    parts=['<svg xmlns="http://www.w3.org/2000/svg" width="800" height="525" viewBox="0 0 800 525">',
           '<rect x="4" y="4" width="792" height="515" rx="12" fill="#F8FBFC" stroke="#235A85" stroke-width="2"/>',
           f'<text x="24" y="32" font-size="20" font-family="Arial" fill="#183047">dbGaP universe: {v["universe"]:,} accessions</text>']
    for x,y,r,c,label,lx,ly in [(275,201,137,'#235A85','SRA',173,70),(471,201,137,'#137D83','BDC',583,70),(373,341,137,'#BA7D24','AnVIL',373,502)]:
        parts.append(f'<circle cx="{x}" cy="{y}" r="{r}" fill="{c}" fill-opacity=".10" stroke="{c}" stroke-width="2"/>')
        parts.append(f'<text x="{lx}" y="{ly}" text-anchor="middle" font-family="Arial" font-size="23" fill="{c}" font-weight="bold">{label}</text>')
    places={'100':(217,195),'010':(532,195),'001':(373,430),'110':(373,141),'101':(285,310),'011':(461,310),'111':(373,264),'000':(79,458)}
    for bits,(x,y) in places.items():
        n=v['counts'].get(bits);shown='?' if n is None else f'{n:,}'
        parts.append(f'<text x="{x}" y="{y}" text-anchor="middle" font-family="Arial" font-size="26" font-weight="bold" fill="#183047">{shown}</text>')
    parts.append('<text x="23" y="425" font-size="15" font-family="Arial" fill="#607487">None of three</text></svg>')
    return ''.join(parts)
s=add('Repository intersections within dbGaP','SRA, BioData Catalyst and AnVIL are overlapping listings/services, not necessarily full copies',
    f'{source_label}. Exact counts use {v["known"]:,} fully known roots; {v["unknown"]:,} unknown excluded from regions. Circle areas are schematic.',
    notes='The rectangle is dbGaP, not a fourth independent circle. Only dbGaP-root-linked footprints are included; these are not whole-repository totals. SRA means at least one consent group available. BDC and AnVIL are explicit listings. Unknown membership is not the outside-all-three region.')
svg_image(s,venn_svg(v),48,151,825,548,'repository_venn')
for j,key in enumerate(v['names']):
    m=v['marginals'][key];card(s,integer(m['positive']),key+'-linked accessions',926,169+j*147,466,119,f"{m['known']:,} known; {m['unknown']:,} unknown")
# 8
b=R['bdc'];s=add('BDC-listed vs not BDC-listed accessions',f"{b['bdc_n']:,} BDC-listed | {b['other_n']:,} not listed | {b['unknown']:,} exposure unknown",
    f'{source_label}. Denominators are outcome-specific. pp = percentage points.',
    notes='Structured listing is not proof every file, consent group or version is stored at BDC. Differences are observational and may reflect program selection, cohort maturity, study size, consent, parent/substudy structure and curation. Current listing can postdate the observed outputs.')
rows=[]
for row in b['rows']:
    form=lambda p,n: f'{p/n*100:.1f}% ({p:,}/{n:,})' if n else 'Not measured'
    rows.append([row['label'],form(row['bdc_positive'],row['bdc_n']),form(row['other_positive'],row['other_n']),
                 '--' if row['difference_pp'] is None else f"{row['difference_pp']:+.1f} pp"])
table(s,['Recorded signal','BDC-listed','Not listed','Difference'],rows,48,172,[485,310,310,239],53,19)
note(s,'Why might the profiles differ? Study age, program mix, data type, consent restrictions, and metadata completeness.',681,17)
# 11. Original slide 9, expanded rather than moved ahead of the descriptive findings.
s=add('Data sources, assumptions and confounders','A collects; B analyzes; C renders | source-specific units and incomplete histories remain visible',
    'Preliminary, cross-sectional discussion. Source hashes and full field definitions accompany the saved results.',
    notes='Data files used in this build:\n'+'\n'.join(i['key']+': '+i['path'] for i in R.get('input_provenance',[]))+
    '\n'+json.dumps(R.get('methods',{}),indent=2)+
    '\nAn HTTP failure is not a negative observation. Different root, version, participant-set, consent-group, PMID, requester and NCT units are retained. The descriptive both/neither comparison excludes mixed signal groups. It does not measure approval-to-use conversion or estimate a platform intervention effect. Human publication provenance and actual file-use telemetry are not present.')
source_rows=[
 ['FHIR / public MeSH / pinned XML','Accessions, v/p; independent native descriptors'],
 ['Study counts / page records','Variables; study attribution and explicit NIH mentions'],
 ['SSTR / OBI assay crosswalk','Loaded subjects/samples, consent names/codes, hierarchy'],
 ['Approved-requester records','WLID, organization, initial approval; checkpoints'],
 ['Study-PMID links + PubMed','Linked literature, metadata and publication year'],
 ['NIH iCite','RCR and scoring eligibility; unique PMID'],
 ['Authorized Access API','SRA consent availability, BDC/AnVIL listing, collections'],
 ['NCT references + ClinicalTrials.gov','Recorded references; may be publication-mediated'],
 ['FTP artifact / local ledger','Named discovery proxy, not true release date']]
table(s,['Saved data source','Used for'],source_rows,48,154,[259,477],49,16)
rect(s,815,154,577,229,C['light']);txt(s,'Assumptions and definitions',833,170,541,31,22,bold=True)
txt(s,'Known negatives differ from failed lookups.\nA root is not an independent cohort.\nSSTR supplies subject and sample counts.\nConsent numbers require local name mappings.\nLinked papers need not be secondary reuse.\nSankey links show co-occurrence, not flow.',833,214,541,160,18)
rect(s,815,399,577,243,'#FBF5EA');txt(s,'Confounders and selection',833,415,541,31,22,bold=True)
txt(s,'Study age and unequal follow-up; program/IC;\ncohort size, data type and phenotype depth;\nconsent restrictions and curation intensity;\nrelated accessions and shared papers;\ncurrent metadata measured after outcomes;\ntwo-group selection; unequal annotation depth.',833,458,541,170,18)
txt(s,'Version matrix: actual highest observed v/p pair per root, not the number of retained releases. Consent labels can overlap.\nSlide 6 is descriptive; semantic bridges are not exact equivalence. Date proxy is not verified age; current-year growth is not annualized.',48,662,1344,62,17,C['muted'])
# 10
s=add('Earlier studies, open questions, discussion','Historical reference points clarify the units; they are not a matched comparison group',
    'Preliminary descriptive discussion. No platform-effect, independent-reuse, patient-benefit or storage-policy conclusion.',
    notes='Sources are primary publications; their populations, units, time windows and ascertainment differ. The current corpus cannot be divided into original-team and independent reuse from first-event timing. Late demand requires actual release dates, complete histories and censoring. Detailed references and links are retained in speaker notes.')
refs={r['key']:r for r in R.get('references',[])}
for j,key in enumerate(['paltoo','cabili','coady']):
    ref=refs[key];x=48+j*453;rect(s,x,157,430,263,C['light'])
    txt(s,ref['label'],x+17,174,396,55,20,C['blue'],True,link=ref['url'])
    txt(s,ref['finding'],x+17,235,396,95,21)
    txt(s,ref['boundary'],x+17,342,396,67,16,C['muted'])
questions=[('What contributes to visible use?','Do DAR timing and investigator breadth add information beyond study age and prior publications?'),
 ('Who is using the data?','Which papers are original-team outputs, secondary analyses by collaborators, or independent reuse?'),
 ('What does cross-listing change?','Does BDC enable new activity, or mainly host already mature and well-used resources?'),
 ('Can quiet data become useful later?','What is the chance of first demand after year three, using true release dates and complete follow-up?')]
for j,(head,body) in enumerate(questions):
    x=48+(j%2)*682;y=449+(j//2)*135
    txt(s,head,x,y,655,33,23,bold=True);txt(s,body,x,y+38,650,78,19)
# Detailed paper-method report and tables are written by B, not recomputed here.
assert len(SLIDES)==14

# Serialize the common layout into offline HTML and editable PowerPoint shapes/text.
def uri(path):
    p=Path(path);mime='image/svg+xml' if p.suffix=='.svg' else 'image/png'
    return 'data:'+mime+';base64,'+base64.b64encode(p.read_bytes()).decode()
def html_element(e):
    style=f"left:{e['x']}px;top:{e['y']}px;width:{e['w']}px;height:{e['h']}px;"
    if e['kind']=='rect':return f'<div style="{style}background:{e["fill"]};'+(f'border:1px solid {e["border"]};' if e.get('border') else '')+'"></div>'
    if e['kind']=='image':return f'<img alt="Figure" style="{style}object-fit:contain" src="{uri(e["path"])}">'
    style+=f"font-size:{e['size']}px;color:{e['color']};font-weight:{'700' if e['bold'] else '400'};text-align:{e['align']};white-space:pre-wrap;line-height:1.18;"
    content=esc(e['text'])
    if e.get('link'):content=f'<a href="{esc(e["link"])}" target="_blank" rel="noopener">{content}</a>'
    return f'<div class="text" style="{style}">{content}</div>'
sections=[]
for i,s in enumerate(SLIDES):
    notes=s['notes']+'\n\n'+ '\n'.join(r['label']+' | '+r['url'] for r in R.get('references',[]))
    sections.append(f'<section class="slide {"active" if i==0 else ""}" id="s{i+1}">'+''.join(html_element(e) for e in s['elements'])+f'<aside>{esc(notes)}</aside></section>')
CSS='*{box-sizing:border-box}body{margin:0;background:#dce4eb;font-family:Arial,sans-serif}.stage{position:absolute;width:1440px;height:810px;transform-origin:0 0}.slide{position:absolute;inset:0;display:none;overflow:hidden}.slide.active{display:block}.slide>div,.slide>img{position:absolute}.text{overflow:hidden}a{color:inherit;text-decoration:underline}aside{display:none}nav{position:fixed;right:8px;bottom:4px;z-index:10}button{padding:5px 10px}#notes{display:none;position:fixed;inset:15%;background:#fff;padding:25px;overflow:auto;z-index:20;white-space:pre-wrap}#notes.open{display:block}@media print{@page{size:1440px 810px;margin:0}body{background:white}.stage{position:static;transform:none!important;height:auto}.slide{position:relative;display:block;width:1440px;height:810px;break-after:page}nav,#notes{display:none!important}}'
JS="""let i=0;const ss=[...document.querySelectorAll('.slide')];function fit(){let z=Math.min(innerWidth/1440,innerHeight/810);let d=document.querySelector('.stage');d.style.transform=`scale(${z})`;d.style.left=(innerWidth-1440*z)/2+'px';d.style.top=(innerHeight-810*z)/2+'px'}function go(n){i=Math.max(0,Math.min(ss.length-1,n));ss.forEach((s,j)=>s.classList.toggle('active',i===j));document.querySelector('#notes').classList.remove('open')}function notes(){let n=document.querySelector('#notes');n.textContent=ss[i].querySelector('aside').textContent;n.classList.toggle('open')}addEventListener('resize',fit);addEventListener('keydown',e=>{if(['ArrowRight','PageDown'].includes(e.key))go(i+1);if(['ArrowLeft','PageUp'].includes(e.key))go(i-1);if(e.key.toLowerCase()==='n')notes();if(e.key==='Escape')document.querySelector('#notes').classList.remove('open')});fit();"""
htmlpath=OUT/'dbgap_preliminary_14_slides.html'
htmlpath.write_text('<!doctype html><html><head><meta charset="utf-8"><title>dbGaP | preliminary discussion</title><style>'+CSS+'</style></head><body><main class="stage">'+''.join(sections)+'</main><nav><button onclick="go(i-1)">Previous</button><button onclick="notes()">Notes</button><button onclick="go(i+1)">Next</button></nav><div id="notes"></div><script>'+JS+'</script></body></html>')
pptxpath=None
if EXPORT_POWERPOINT:
    from pptx import Presentation
    from pptx.util import Inches,Pt
    from pptx.dml.color import RGBColor
    from pptx.enum.shapes import MSO_SHAPE
    from pptx.enum.text import PP_ALIGN,MSO_ANCHOR
    import cairosvg
    prs=Presentation();prs.slide_width=Inches(13.333333);prs.slide_height=Inches(7.5)
    def length(px):return Inches(px/108)
    def rgb(v):return RGBColor.from_string(v.lstrip('#'))
    cache={}
    for s in SLIDES:
        sl=prs.slides.add_slide(prs.slide_layouts[6])
        for e in s['elements']:
            x,y,w,h=[length(e[k]) for k in ['x','y','w','h']]
            if e['kind']=='rect':
                sh=sl.shapes.add_shape(MSO_SHAPE.RECTANGLE,x,y,w,h);sh.fill.solid();sh.fill.fore_color.rgb=rgb(e['fill']);sh.line.fill.background()
            elif e['kind']=='text':
                tf=sl.shapes.add_textbox(x,y,w,h).text_frame;tf.clear();tf.margin_left=tf.margin_right=tf.margin_top=tf.margin_bottom=0
                tf.word_wrap=True;tf.vertical_anchor=MSO_ANCHOR.TOP
                for q,line in enumerate(e['text'].split('\n')):
                    p=tf.paragraphs[0] if q==0 else tf.add_paragraph();p.space_before=Pt(0);p.space_after=Pt(0);p.line_spacing=1.05
                    p.alignment={'left':PP_ALIGN.LEFT,'center':PP_ALIGN.CENTER,'right':PP_ALIGN.RIGHT}[e['align']]
                    run=p.add_run();run.text=line;run.font.name='Arial';run.font.size=Pt(e['size']*2/3);run.font.bold=e['bold'];run.font.color.rgb=rgb(e['color'])
                    if e.get('link'):run.hyperlink.address=e['link']
            else:
                p=Path(e['path'])
                if p.suffix.lower()=='.svg':
                    if str(p) not in cache:cache[str(p)]=cairosvg.svg2png(url=str(p),output_width=max(1500,int(e['w']*2)))
                    content=io.BytesIO(cache[str(p)])
                else:content=str(p)
                # Preserve aspect ratio within the image rectangle.
                from PIL import Image
                if isinstance(content,io.BytesIO):iw,ih=Image.open(content).size;content.seek(0)
                else:iw,ih=Image.open(content).size
                ratio=min(e['w']/iw,e['h']/ih);ww=iw*ratio;hh=ih*ratio
                sl.shapes.add_picture(content,length(e['x']+(e['w']-ww)/2),length(e['y']+(e['h']-hh)/2),length(ww),length(hh))
        sl.notes_slide.notes_text_frame.text=s['notes']+'\n\nSources:\n'+'\n'.join(r['label']+'\n'+r['url'] for r in R.get('references',[]))
    assert len(prs.slides)==14
    pptxpath=OUT/'dbgap_preliminary_14_slides.pptx';prs.save(pptxpath)
(OUT/'presentation_manifest.json').write_text(json.dumps({'slide_count':14,'summary_sha256':digest(BASE/'summary.json'),'input_signature':R['input_signature'],
        'mode':mode,'html_sha256':digest(htmlpath),'pptx_sha256':digest(pptxpath) if pptxpath else None,'notes':'No data acquisition; charts are graphics, text/shapes editable.'},indent=2))
import shutil
_paper=BASE/R.get('replication_report','paper_replication_report.html')
if _paper.is_file():
    shutil.copy2(_paper,OUT/'paper_replication_report.html')
    for _p in BASE.glob('rep_*.csv'):shutil.copy2(_p,OUT/_p.name)
print('Created 14 slides:');print(htmlpath)
if pptxpath:print(pptxpath)
if DOWNLOAD_IN_COLAB:
    try:
        from google.colab import files
        files.download(str(pptxpath or htmlpath))
    except ImportError:pass
